# 9. CPU and GPU paths

rang runs on the CPU everywhere. One kernel, the w-stacking gridder and
its FFT, also has a GPU path through `wgpu` (Vulkan, Metal, DX12; no CUDA
toolkit), behind the cargo feature `gpu`, off in the published wheels
({gh}`research/x2-wgpu.md`). The rule (`AGENTS.md`, {gh}`research/vision.md`
section 4): every GPU path has a CPU path that gives the same result within
a written tolerance, and the CPU path is the reference.

This notebook shows which parts run where, makes the same image on each
path that this machine has, and checks them against each other with the
frozen tolerances. On a machine without the feature or a GPU (the
documentation build, a laptop) the GPU cells print a skip line and the
measured X2 figures are shown instead. It runs on the public fixture
(scan 9 of `outer_scans.zip`, four MeerKAT antennas, 4096 channels, as in
notebooks 1-4). About 30 s.

**Learning goals.** After this notebook you can

- say which parts of rang have a GPU path, and which settings select it;
- check whether your build and machine can use it;
- compare two paths of the same computation with a tolerance relative to
  the image peak, and say why bit-identical is not the bar between devices;
- say why the f32 GPU path is not a stand-in for the CPU, and why results
  on the CPU do not depend on the thread count.

In [ ]:
%matplotlib inline
import hashlib
import json
import os
import subprocess
import sys
import time

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd
from astropy.io import fits

from rang import core_grid
from rang.notebook import data, gains, image

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

CACHE = data.default_cache_dir()
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)
ds = data.load_fixture(cache_dir=CACHE)
sol = gains.solve(ds)
print({k: v for k, v in data.summary(ds).items() if k in ("antennas", "baselines", "channels", "rows", "integrations")})

## 1. What runs where

| Part | CPU | GPU | Selected by | Report |
| --- | --- | --- | --- | --- |
| w-stacking gridder and FFT (`rang.core_grid.invert`; notebooks 1-5, the campaigns) | yes, the reference | f64 (gated), f32 (experimental) | `RANG_GRIDDER_DEVICE=gpu`, `RANG_GRIDDER_GPU_PRECISION`, or `device=`, `precision=` | {gh}`research/x2-wgpu.md` |
| exact w-gridder (`gridder="wgrid"`) | yes | no | `gridder=` | {gh}`research/df2b-wgrid.md` |
| GeMClean (`gemclean`, the image stage of `rang-core run`) | yes, with AVX2/AVX-512 kernels | no | `RANG_GEMCLEAN_SIMD` caps the instruction set | {gh}`research/gemclean.md` |
| flagging, calibration solves, apply | yes | no | – | {gh}`research/finish-plan.md` |

So `rang-core run` and `gemclean` run on the CPU only. A GPU request on a
build without the feature, or with no usable adapter, raises a
`RuntimeError` naming the reason; it never falls back silently to the CPU
(a silent fallback would make a timing or a result mean something else).

What this build and machine have:

In [ ]:
compiled = core_grid.gpu_compiled()
available, adapter = core_grid.gpu_available() if compiled else (False, "the extension was built without the cargo feature `gpu`")
print(f"GPU gridder compiled in: {compiled}")
print(f"usable adapter: {available} ({adapter})")
print(f"RANG_GRIDDER_DEVICE = {os.environ.get('RANG_GRIDDER_DEVICE', 'unset (cpu)')}, RAYON_NUM_THREADS = {os.environ['RAYON_NUM_THREADS']}")

To build the extension with the GPU gridder from a checkout:

```sh
maturin develop --release --features gpu
```

The f64 path needs an adapter with `SHADER_F64` (X2 used an RTX 4070 Ti
through Vulkan); Apple GPUs (Metal) have no f64, so only the f32 path could
run there, and software adapters such as llvmpipe are refused. `RANG_GPU_ADAPTER` picks an
adapter by a substring of its name, `RANG_GPU_MEMORY_MB` sets the device
memory per inversion (256 MB by default, batching only, results unchanged).

## 2. The reference: the CPU image

The corrected data of notebook 3's solve, imaged as notebook 4 does
(Briggs 0.5, 512² pixels of 8″, 64 w-layers):

In [ ]:
start = time.perf_counter()
cpu = image.image(ds, sol, device="cpu")
cpu_s = time.perf_counter() - start
peak = np.abs(cpu["dirty"]).max()
print(f"CPU, {os.environ['RAYON_NUM_THREADS']} threads: {cpu_s:.2f} s (this machine, one run: not a benchmark); dirty peak {peak:.4f} Jy/beam")
image.image_figure(cpu)

## 3. How to compare two paths

Two devices add the same numbers in a different order, so their results
differ by rounding; bit-identity across devices is the wrong bar. The bar is
a tolerance written before the measurement, relative to the image peak:

- **f64 GPU against the CPU:** max |GPU − CPU| / peak ≤ 1e-10 (X2's
  criterion 4). Measured: at most 2.2e-15.
- **Either path against WSClean** (the oracle): task A's pixel tolerances,
  dirty 8.8e-6 and PSF 2.8e-6 of the peak.

The same comparison applies to any two implementations. A first example
that runs everywhere: the w-stacking gridder against the exact w-gridder,
two different algorithms for the same image. They are not expected to agree
to rounding: w-stacking approximates the w-term with 64 layers and a
7-tap kernel; the exact gridder works to `epsilon`.

DF2b measured exactly this ({gh}`research/df2b-wgrid.md`): against the
w-stacking images the exact gridder moves pixels by 17-32 % of the peak at
the image edge and at most 0.56 % in the central half on four-station
images. The difference is w-stacking's aliasing near the edge, not an
error of either path, and no hypothesis call of the campaigns changed.
The exact gridder agrees with WSClean's own wgridder within 4.0e-7 of the
peak. Two algorithms are compared region by region, with the tolerance
each one was frozen with; two devices running one algorithm are compared
at rounding level, as below.

In [ ]:
def compare(a, b, central=False):
    out = {}
    for k in ("dirty", "psf"):
        x, y = a[k], b[k]
        n0, n1 = y.shape
        part = (slice(n0 // 4, 3 * n0 // 4), slice(n1 // 4, 3 * n1 // 4)) if central else (slice(None), slice(None))
        out[f"{'PSF' if k == 'psf' else k}: max |a − b| / peak" + (", central half" if central else "")] = float(np.abs(x - y)[part].max() / np.abs(y).max())
    return out


start = time.perf_counter()
exact = image.image(ds, sol, gridder="wgrid")
print(f"exact w-gridder: {time.perf_counter() - start:.2f} s")
pd.Series({**compare(exact, cpu), **compare(exact, cpu, central=True)}, name="wgrid against wstack").to_frame()

## 4. The CPU path does not depend on the thread count

The CPU gridder's result does not depend on how many threads run it: X2's
criterion 6 found every comparator image bit-identical at 1 and at 16
threads. Here the same image is made in a fresh
process with one thread and its SHA-256 compared:

In [ ]:
def digest(img):
    return hashlib.sha256(np.ascontiguousarray(img["dirty"]).tobytes() + np.ascontiguousarray(img["psf"]).tobytes()).hexdigest()


one_thread = f"""
import hashlib, numpy as np
from rang.notebook import data, gains, image
ds = data.load_fixture(cache_dir={str(CACHE)!r})
img = image.image(ds, gains.solve(ds), device="cpu")
print(hashlib.sha256(np.ascontiguousarray(img["dirty"]).tobytes() + np.ascontiguousarray(img["psf"]).tobytes()).hexdigest())
"""
p = subprocess.run([sys.executable, "-c", one_thread], env={**os.environ, "RAYON_NUM_THREADS": "1"}, capture_output=True, text=True, check=True)
single = p.stdout.strip().splitlines()[-1]
print(f"{os.environ['RAYON_NUM_THREADS']} threads: {digest(cpu)[:16]}…")
print(f"1 thread : {single[:16]}…")
print("bit-identical:", single == digest(cpu))

## 5. The GPU paths

With the feature and an adapter, the f64 path is checked against the CPU
at X2's bound and the f32 path is reported against task A's tolerances.
Without them the cell prints its skip line and the next one shows X2's
measurement.

In [ ]:
gpu_rows = []
if not available:
    print(f"skipped: no GPU path here ({adapter})")
else:
    for precision, bound in (("f64", 1e-10), ("f32", 8.8e-6)):
        start = time.perf_counter()
        gpu = image.image(ds, sol, device="gpu", precision=precision)
        seconds = time.perf_counter() - start
        c = compare(gpu, cpu)
        gpu_rows.append({"path": f"GPU {precision}", "seconds": round(seconds, 3), **{k: f"{v:.1e}" for k, v in c.items()},
                         "bound (dirty)": bound, "within": c["dirty: max |a − b| / peak"] <= bound})
    gpu_rows.insert(0, {"path": "CPU", "seconds": round(cpu_s, 3)})
pd.DataFrame(gpu_rows) if gpu_rows else None

X2's measurement on an RTX 4070 Ti from native Windows (Vulkan, driver
621.01; {gh}`research/x2-wgpu.md`, archive
{gh}`research/results/x2-wgpu.json`), per image, wall time including
transfers:

| Set | CPU 1 thread | CPU 16 threads | GPU f64 | GPU f32 | f64 vs CPU, max \|Δ\| / peak |
| --- | ---: | ---: | ---: | ---: | ---: |
| K, 6 four-station arms | 0.82 s | 0.30 s | 0.12 s (2.5× over 16 threads) | 0.077 s | 2.2e-15 |
| C, 15 four-station jobs | 0.80 s | 0.30 s | 0.092 s (3.3×) | 0.060 s | 1.2e-15 |
| whole array, 23.8 M visibilities | 7.6 s | 3.1 s | 2.5 s | – | 1.0e-15 |

Every f64 image passed the comparators against WSClean (tasks C, I, K) as
the CPU path does, and was bit-identical over repeats and batchings. One
frozen clause failed: X2 asked all eight gridder counters to be exactly
equal, and the PSF normalisation differs by 4.4e-16 relative.

**The f32 path fails one comparator**: it passes tasks C and I, but on task
K the residual metrics (dirty − peak × PSF) differ by 1.2-3.5e-5 relative
against a tolerance of 8.8e-6, because the subtraction amplifies f32
rounding. It stays experimental and is never the default.

On the whole array, 1.1 s of the GPU's 2.5 s is a serial pass on the CPU
(WSClean's counters) that both paths share: the GPU pays only where the
gridding dominates. Hence the rule: a GPU path is added where it pays, and
only with its CPU twin and a written tolerance.

## 6. CPU instruction sets: GeMClean's SIMD kernels

GeMClean's minor-cycle kernels dispatch at run time to AVX-512, AVX2 or
scalar code; `RANG_GEMCLEAN_SIMD` caps the choice and `-report.json`
records the tier used (`run.simd`). The tiers are only promised to agree
to rounding, because the gather sum may change its order
({doc}`../../cli`, "Environment variables"); GeMClean's measurements found
the SIMD kernels 24-39 % faster in the minor loop and no faster in whole-run
wall time ({gh}`research/gemclean.md`). The same small CLEAN on the
fixture MS at the best tier and at scalar:

In [ ]:
ms = data.extract_fixture(data.find_fixture() or data.download_fixture(CACHE), CACHE)
work = CACHE / "cpu-and-gpu"
work.mkdir(parents=True, exist_ok=True)
models = {}
for tier in (None, "scalar"):
    env = {**os.environ}
    env.pop("RANG_GEMCLEAN_SIMD", None)
    if tier:
        env["RANG_GEMCLEAN_SIMD"] = tier
    name = f"simd-{tier or 'best'}"
    argv = [sys.executable, "-m", "rang.gemclean", "-name", str(work / name), "-size", "512", "512", "-scale", "8asec",
            "-weight", "briggs", "0.5", "-niter", "500", "-j", os.environ["RAYON_NUM_THREADS"], "-no-update-model-required", str(ms)]
    start = time.perf_counter()
    subprocess.run(argv, env=env, capture_output=True, text=True, check=True)
    report = json.loads((work / f"{name}-report.json").read_text(encoding="utf-8"))
    print(f"{name}: tier {report['run']['simd']} (CPU offers {report['simd_detected']}), {report['run']['components']} components, {time.perf_counter() - start:.1f} s")

    models[name] = {k: np.squeeze(fits.getdata(work / f"{name}-{k}.fits")).astype(float) for k in ("model", "residual")}
best, scalar = models["simd-best"], models["simd-scalar"]
for k in ("model", "residual"):
    diff = np.abs(best[k] - scalar[k]).max() / np.abs(scalar[k]).max()
    print(f"{k}: max |best − scalar| / peak = {diff:.1e}, identical: {np.array_equal(best[k], scalar[k])}")

## Exercises

1. Set `RANG_GRIDDER_DEVICE=gpu` before starting the kernel on a build
   without the feature. What does `image.image` raise, and where would a
   silent CPU fallback have misled you?
2. Image with 32 and 128 w-layers (`w_layers=`). How does the difference
   against the exact w-gridder change?
3. With a GPU build, set `RANG_GPU_MEMORY_MB=64`. Does the image change?
   Does the time?
4. Time `image.image` at 1, 4 and 16 threads under
   `tools/heavy_lock.py run --` (an otherwise idle host). Where does the
   speed-up stop, and why does a timing on a shared host mean little?

## Where next

- {doc}`04-imaging`: the image this notebook computes on each path.
- {doc}`08-reports-and-image-quality`: judging an image once it is made.
- {gh}`research/x2-wgpu.md`: the GPU gridder's protocol, frozen before
  any GPU code, and every measured figure; {gh}`research/gemclean-speed.md`
  for GeMClean's speed work on the CPU.